In [1]:
from selenium import webdriver
from selenium.webdriver.support.select import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

#Bibliotecas de Sistema
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

In [2]:
#Bibliotecas de Sistema
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

pasta_downloads = r"C:\Users\dodonin\Downloads"
navegador = eproc.novo_browser(pasta_downloads)

#configura variáveis
username = "dodonin"
password = keyring.get_password("eproc", username)
pyotop_code = "GJRGIYTCGBSGKYTEHE2TOZRUGFQTMMRQ"

eproc.login_no_eproc(navegador, username, password, pyotop_code)

Driver do Eproc importado


In [4]:
processo = 50046188820248210069
perfil = "SRD1CIV"

In [5]:
eproc.entrar_no_perfil(navegador, perfil)

Perfil carregado: SRD1CIV


In [6]:
eproc.entrar_no_processo(navegador, processo)

Página do processo carregada com sucesso.


In [7]:
def pegaDoc(driver, doc):    
    # Localiza o elemento com código
    elemento = driver.find_element(By.PARTIAL_LINK_TEXT, str(doc))
    # Cria uma cadeia de ações e move o mouse até o elemento
    actions = ActionChains(driver)
    actions.move_to_element(elemento).perform()
    time.sleep(4)
    # Pega o texto
    pyautogui.click(1000, 600)
    time.sleep(0.5)  # Pequena pausa para garantir que o foco esteja correto
    pyautogui.hotkey('ctrl', 'a')
    time.sleep(0.2)  # Pequena pausa para segurança
    pyautogui.hotkey('ctrl', 'c')
    time.sleep(0.2)  # Dá tempo do sistema copiar para a área de transferência
    conteudo = pyperclip.paste()
    pyautogui.click(1000, 600)
    pyautogui.hotkey('f5')
    time.sleep(3)  # Pequena pausa para segurança
    #devolve o conteudo
    print(conteudo)

In [8]:
links = navegador.find_elements(By.CLASS_NAME, "td-evento")

for link in links:
    id_link=link.get_dom_attribute("id")    
    texto_link = link.text.split('\n')[0]
    tr_element = link.find_element(By.XPATH, "./ancestor::tr")
    # Pega o texto da segunda coluna (td[2]) de tr_element
    evento_text = tr_element.find_element(By.XPATH, './td[2]').text    
    print(id_link + " , "+ texto_link+ " , "+ evento_text )

    #pegaDoc(navegador, link.text)

tdEvento14Doc1 ,   PET1 , 14 
tdEvento14Doc2 ,   OUT2 , 14 
tdEvento14Doc3 ,   OUT3 , 14 
tdEvento8Doc1 ,   DESPADEC1 , 8 
tdEvento6Doc1 ,   PET1 , 6 
tdEvento3Doc1 ,   DESPADEC1 , 3 
 
tdEvento1Doc1 ,   INIC1 , 1 
tdEvento1Doc2 ,   PROC2 , 1 
tdEvento1Doc3 ,   CDA3 , 1 


In [5]:
conn = sqlite3.connect("movimentos.db")
cursor = conn.cursor()

In [16]:
eventos = [el for el in navegador.find_elements(By.CLASS_NAME, "infraEventoDescricao") if el.tag_name == "label"]

for evento in eventos:
    texto_evento = evento.text.split('\n')[0]
    tr_element = evento.find_element(By.XPATH, "./ancestor::tr")
    # Pega o texto da segunda coluna (td[2]) de tr_element
    evento_text = tr_element.find_element(By.XPATH, './td[2]').text
    conn = sqlite3.connect("movimentos.db")
    cursor = conn.cursor()
    cursor.execute(
        "SELECT 1 FROM movimentos WHERE processo = ? AND evento = ? LIMIT 1",
        (str(processo), int(evento_text))
    )
    existe = cursor.fetchone() is not None
    if not existe:
        cursor.execute(
            "INSERT INTO movimentos (processo, evento, vara, descricao) VALUES (?, ?, ?, ?)",
            (str(processo), int(evento_text), perfil, texto_evento)
        )
        conn.commit()


In [14]:
conn = sqlite3.connect("movimentos.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS movimentos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    processo TEXT,
    vara TEXT,
    evento INTEGER,
    descricao TEXT,
    documento TEXT,
    resumo TEXT,
    encaminhamento TEXT
)
""")

conn.commit()
conn.close()

In [7]:
conn.close()

In [9]:

def pega_texto_documento(driver, documento):
    pyautogui.click(1000, 600)
    time.sleep(0.2)  # Pequena pausa para segurança
    # Localiza o elemento com código
    elemento = driver.find_element(By.ID, str(documento))
    # Cria uma cadeia de ações e move o mouse até o elemento
    actions = ActionChains(driver)
    actions.move_to_element(elemento).perform()
    time.sleep(4)
    # Pega o texto
    pyautogui.click(1000, 600)
    time.sleep(0.5)  # Pequena pausa para garantir que o foco esteja correto
    pyautogui.hotkey('ctrl', 'a')
    time.sleep(0.2)  # Pequena pausa para segurança
    pyautogui.hotkey('ctrl', 'c')
    time.sleep(0.2)  # Dá tempo do sistema copiar para a área de transferência
    conteudo = pyperclip.paste()
    pyautogui.click(1000, 600)
    pyautogui.hotkey('f5')
    time.sleep(3)  # Pequena pausa para segurança
    #devolve o conteudo
    return conteudo

In [11]:
def docum(navegador, link):
    pyautogui.click(1000, 600)    
    actions = ActionChains(navegador)
    time.sleep(1)
    actions.move_to_element(link).perform()
    time.sleep(4)
    return

In [58]:
for link in links:
    docum(navegador, link)

In [ ]:
documento = "tdEvento14Doc1"
pyautogui.click(1000, 600)
time.sleep(0.2)  # Pequena pausa para segurança
elemento = navegador.find_element(By.ID, str(documento))
# Cria uma cadeia de ações e move o mouse até o elemento
actions = ActionChains(navegador)
actions.move_to_element(elemento).perform()
time.sleep(4)

In [39]:
print(pega_texto_documento(navegador, "tdEvento14Doc1"))

Ir para conteúdo Ir para menu Pesquisa processual text_increase text_decrease contrast  Libras Acessibilidadeclose
menu
Painel Inicial
RS

SRD1CIV/DIRETOR DE SECRETARIA
  
PDPJ - Marketplace
home
all_inbox
Nº de processo
account_circle
Pesquisar no Menu (Alt + m)
Menu Textual
AJG
arrow_drop_down
Alvará Eletrônico Automatizado
arrow_drop_down
Atendimento e Tutoriais
Audiência
arrow_drop_down
Bens Associados
arrow_drop_down
Bloqueios/impedimentos dos Peritos
Certidões
arrow_drop_down
Cisão/Desmembramento de Processo
Consulta Processual
arrow_drop_down
Custas
arrow_drop_down
Depósitos Judiciais
arrow_drop_down
Execução Fiscal
arrow_drop_down
Execução Penal
arrow_drop_down
Gerenciamento de Advogados/Conciliadores
arrow_drop_down
Gerenciamento de Entidades
arrow_drop_down
Gerenciamento de Feriados e Suspensões
arrow_drop_down
Gerenciamento de Partes
arrow_drop_down
Gerenciamento de Processos Físicos
arrow_drop_down
Gerenciamento de Processos Relacionados
arrow_drop_down
Gerenciamento do Pla